# Consensus Sequences

Generate consensus sequences from multiple sequence alignments of all unique haplotypes within a given transcript.

Sources:
- https://help.geneious.com/hc/en-us/articles/360044627712-Which-multiple-alignment-algorithm-should-I-use
- https://biopython.org/docs/1.74/api/Bio.Align.Applications.html

In [5]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import os
# only load this one time per session
if 'NOTEBOOK_INITIALIZED' not in globals():
    os.chdir(os.path.dirname(os.path.abspath('.')))
    NOTEBOOK_INITIALIZED = True

import src.utils as utils
import src.config as config
import src.haplosaurus as hs
import src.ESM as ESM
import src.ESM_predict as ESMp
import src.gprofiler as gp
import src.vep_pipeline as vp
import src.vep_analysis as va
import src.vep_metrics as vm
import src.proteingym as pg 
import src.ensembl_rest as er
import src.biopython as bp

pd.set_option('display.max_columns', None)

List the trancripts IDs you want to generate consensus sequences for.

In [43]:
tx_ids = hs.list_haplotypes()[:10]

Found haplotypes of 47325 transcripts in: '/home/schilder/.cache/ensembl_rest/haplotypes'


Generate the FASTA files for each transcript and align them using ClustalOmega.

In [63]:
from tqdm import tqdm
from src._ClustalOmega import ClustalOmegaCommandline

force = False
alignment_paths = {}
for tx_id in tqdm(tx_ids, desc="Aligning haplotypes"):

    # Create the input FASTA file (unaligned)
    fasta_paths = hs.haplotypes_to_fasta(tx_ids=tx_id,
                                         merge=False, 
                                         verbose=False)
    fasta_path = fasta_paths[list(fasta_paths.keys())[0]]

    # Create the output file path
    out_file = fasta_path.replace(os.sep+"split"+os.sep, os.sep+"clustalo"+os.sep)
    alignment_paths[tx_id] = out_file

    if os.path.exists(out_file) and not force:
        continue
    
    os.makedirs(os.path.dirname(out_file), exist_ok=True)

    # Create the ClustalOmega command
    clustalomega_cline = ClustalOmegaCommandline(infile=fasta_path, 
                                                outfile=out_file, 
                                                verbose=False, 
                                                auto=True)

    # Run the ClustalOmega command
    clustalomega_cline()




Aligning haplotypes:   0%|          | 0/10 [00:00<?, ?it/s]

Aligning haplotypes: 100%|██████████| 10/10 [00:00<00:00, 19.50it/s]


Compute consensus sequences from the aligned FASTA files.

In [66]:
from Bio import AlignIO
from Bio.motifs import Motif

identity_threshold=0
consensus_seqs = {}
for tx_id, alignment_path in tqdm(alignment_paths.items(), 
                                  desc="Generating consensus sequences"):

    # Read the alignment using Bio.Align instead of Bio.AlignIO
    alignment = AlignIO.read(alignment_path, format="fasta")

        # Convert to new-style Alignment object
    new_alignment = alignment.alignment

    # Create a Motif object from the alignment with protein alphabet
    motif = Motif(alphabet="ACDEFGHIKLMNPQRSTVWY", 
                  alignment=new_alignment)

    # Get the consensus sequence using the counts property
    consensus_seqs[tx_id] = motif.counts.calculate_consensus(identity=identity_threshold)

Generating consensus sequences: 100%|██████████| 10/10 [00:00<00:00, 24.82it/s]


In [67]:
consensus_seqs

{'ENST00000254854': 'MTACARRAGGLPDPGLCGPAWWAPSLPRLPRALPRLPLLLLLLLLQPPALSAVFTVGVLGPWACDPIFSRARPDLAARLAAARLNRDPGLAGGPRFEVALLPEPCRTPGSLGAVSSALARVSGLVGPVNPAACRPAELLAEEAGIALVPWGCPWTQAEGTTAPAVTPAADALYALLRAFGWARVALVTAPQDLWVEAGRSLSTALRARGLPVASVTSMEPLDLSGAREALRKVRDGPRVTAVIMVMHSVLLGGEEQRYLLEAAEELGLTDGSLVFLPFDTIHYALSPGPEALAALANSSQLRRAHDAVLTLTRHCPSEGSVLDSLRRAQERRELPSDLNLQQVSPLFGTIYDAVFLLARGVAEARAAAGGRWVSGAAVARHIRDAQVPGFCGDLGGDEEPPFVLLDTDAAGDRLFATYMLDPARGSFLSAGTRMHFPRGGSAPGPDPSCWFDPNNICGGGLEPGLVFLGFLLVVGMGLAGAFLAHYVRHRLLHMQMVSGPNKIILTVDDITFLHPHGGTSRKVAQGSRSSLGARSMSDIRSGPSQHLDSPNIGVYEGDRVWLKKFPGDQHIAIRPATKTAFSKLQELRHENVALYLGLFLARGAEGPAALWEGNLAVVSEHCTRGSLQDLLAQREIKLDWMFKSSLLLDLIKGIRYLHHRGVAHGRLKSRNCIVDGRFVLKITDHGHGRLLEAQKVLPEPPRAEDQLWTAPELLRDPALERRGTLAGDVFSLAIIMQEVVCRSAPYAMLELTPEEVVQRVRSPPPLCRPLVSMDQAPVECILLMKQCWAEQPELRPSMDHTFDLFKNINKGRKTNIIDSMLRMLEQYSSNLEDLIRERTEELELEKQKTDRLLTQMLPPSVAEALKTGTPVEPEYFEQVTLYFSDIVGFTTISAMSEPIEVVDLLNDLYTLFDAIIGSHDVYKVETIGDAYMVASGLPQRNGQRHAAEIANMSLDILSAVGTFRMRHMPEVPVRIRIGL